In [26]:
# imports
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import re

In [27]:
out_sensor_data = pd.read_csv('/local/foam/cases/data/case4/period1/sensor_out.csv').sort_values(by='dt')
out_sensor_data["windspeed"] = (out_sensor_data["windspeed"]* 0.44704).round()  # convert from mph to m/s
out_sensor_data["windavg"] = (out_sensor_data["windavg"]* 0.44704).round()

In [18]:
len(out_sensor_data)

47

In [17]:
sorted(out_sensor_data["windspeed"].unique())

[np.float64(4.0),
 np.float64(5.0),
 np.float64(6.0),
 np.float64(7.0),
 np.float64(8.0),
 np.float64(9.0),
 np.float64(10.0),
 np.float64(11.0),
 np.float64(12.0),
 np.float64(13.0),
 np.float64(14.0),
 np.float64(17.0)]

In [28]:
sensor_in_x = 50
sensor_in_y = 46
sensor_in_z = 1

In [29]:
csv_files = [i for i in os.listdir("/local/home/liubov_kurafeeva/CFDaPCR_intheloop/data/simulations") if i.endswith(".csv")]
print(csv_files)

['sim_6_ws_12.96_wd_35.csv', 'sim_1_ws_5.51_wd_35.csv', 'sim_0_ws_4.02_wd_35.csv', 'sim_2_ws_7.00_wd_35.csv', 'sim_9_ws_17.43_wd_35.csv', 'sim_7_ws_14.45_wd_35.csv', 'sim_8_ws_15.94_wd_35.csv', 'sim_5_ws_11.47_wd_35.csv', 'sim_4_ws_9.98_wd_35.csv', 'sim_3_ws_8.49_wd_35.csv']


In [ ]:
# Load simulation data and extract U_Magnitude at sensor_in location for each simulation
simulation_dir = "/local/home/liubov_kurafeeva/CFDaPCR_intheloop/data/simulations"

# Dictionary to store windspeed (from filename) -> U_Magnitude at sensor_in
simulation_lookup = {}

for csv_file in csv_files:
    # Extract windspeed from filename (e.g., sim_0_ws_4.02_wd_35.csv -> 4.02)
    match = re.search(r'ws_([\d.]+)_wd', csv_file)
    if match:
        ws_value = float(match.group(1))
        
        # Load CSV and find the row matching sensor_in coordinates
        df = pd.read_csv(os.path.join(simulation_dir, csv_file))
        
        # Find the point closest to sensor_in coordinates
        sensor_in_row = df[(df['x'] == sensor_in_x) & 
                           (df['y'] == sensor_in_y) & 
                           (df['z'] == sensor_in_z)]
        
        if len(sensor_in_row) > 0:
            u_magnitude = sensor_in_row['U_Magnitude'].values[0]
            simulation_lookup[ws_value] = u_magnitude
        else:
            # Find closest point if exact match not found
            df['distance'] = np.sqrt((df['x'] - sensor_in_x)**2 + 
                                     (df['y'] - sensor_in_y)**2 + 
                                     (df['z'] - sensor_in_z)**2)
            closest_row = df.loc[df['distance'].idxmin()]
            simulation_lookup[ws_value] = closest_row['U_Magnitude']

print(f"Loaded {len(simulation_lookup)} simulations")
print("Windspeed -> U_Magnitude mapping:")
for ws, u_mag in sorted(simulation_lookup.items()):
    print(f"  WS={ws:.2f} m/s -> U_Magnitude={u_mag:.4f}")

In [ ]:
# Create sequences of 12 consecutive out sensor readings
# The 12th value determines which simulation to use for the target (sensor_in value)

SEQUENCE_LENGTH = 12

def get_closest_simulation_ws(ws_value, available_ws):
    """Find the simulation windspeed closest to the given windspeed"""
    return min(available_ws, key=lambda x: abs(x - ws_value))

# Get available simulation windspeeds
available_ws = list(simulation_lookup.keys())
print(f"Available simulation windspeeds: {sorted(available_ws)}")

# Create sequences from out_sensor_data
sequences = []
targets = []

for i in range(len(out_sensor_data) - SEQUENCE_LENGTH + 1):
    # Get 12 consecutive windspeed readings
    seq = out_sensor_data['windspeed'].iloc[i:i+SEQUENCE_LENGTH].values
    
    # The 12th value (last in sequence) determines which simulation to match
    ws_12th = seq[-1]  # Last value in sequence
    
    # Find closest simulation windspeed
    closest_sim_ws = get_closest_simulation_ws(ws_12th, available_ws)
    
    # Target is the U_Magnitude at sensor_in for that simulation
    target = simulation_lookup[closest_sim_ws]
    
    sequences.append(seq)
    targets.append(target)

X = np.array(sequences)
y = np.array(targets)

print(f"Created {len(X)} sequences of length {SEQUENCE_LENGTH}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Build Principal Component Regression (PCR) model
# PCR = PCA for dimensionality reduction + Linear Regression

# Determine optimal number of components (use fewer components than features)
n_components = min(6, SEQUENCE_LENGTH)  # Use up to 6 principal components

pcr_model = Pipeline([
    ('scaler', StandardScaler()),  # Standardize features
    ('pca', PCA(n_components=n_components)),  # Reduce dimensionality
    ('regression', LinearRegression())  # Linear regression on principal components
])

# Fit the model
pcr_model.fit(X_train, y_train)

# Get PCA explained variance
pca = pcr_model.named_steps['pca']
print(f"Number of components: {n_components}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {sum(pca.explained_variance_ratio_):.4f}")

In [ ]:
# Evaluate the model
y_train_pred = pcr_model.predict(X_train)
y_test_pred = pcr_model.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Model Performance:")
print(f"  Training RMSE: {train_rmse:.4f}")
print(f"  Test RMSE: {test_rmse:.4f}")
print(f"  Training R²: {train_r2:.4f}")
print(f"  Test R²: {test_r2:.4f}")

In [ ]:
# Function to predict windspeed at sensor_in given 12 out sensor readings
def predict_sensor_in_windspeed(out_readings):
    """
    Predict the windspeed magnitude at the 'in' sensor location.
    
    Parameters:
    -----------
    out_readings : array-like of shape (12,)
        12 consecutive windspeed readings from the 'out' sensor (in m/s)
    
    Returns:
    --------
    float
        Predicted U_Magnitude at the 'in' sensor location
    """
    out_readings = np.array(out_readings).reshape(1, -1)
    prediction = pcr_model.predict(out_readings)[0]
    return prediction

# Example prediction
example_sequence = X_test[0]
predicted_u_mag = predict_sensor_in_windspeed(example_sequence)
actual_u_mag = y_test[0]

print(f"Example prediction:")
print(f"  Input sequence (12 out sensor readings): {example_sequence}")
print(f"  Predicted U_Magnitude at sensor_in: {predicted_u_mag:.4f}")
print(f"  Actual U_Magnitude at sensor_in: {actual_u_mag:.4f}")

In [ ]:
# Visualize predictions vs actual values
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Predicted vs Actual scatter plot
ax1 = axes[0]
ax1.scatter(y_test, y_test_pred, alpha=0.5, label='Test data')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect prediction')
ax1.set_xlabel('Actual U_Magnitude')
ax1.set_ylabel('Predicted U_Magnitude')
ax1.set_title(f'PCR Model: Predicted vs Actual\nTest R² = {test_r2:.4f}')
ax1.legend()
ax1.grid(True)

# Plot 2: Residuals
ax2 = axes[1]
residuals = y_test - y_test_pred
ax2.scatter(y_test_pred, residuals, alpha=0.5)
ax2.axhline(y=0, color='r', linestyle='--')
ax2.set_xlabel('Predicted U_Magnitude')
ax2.set_ylabel('Residuals')
ax2.set_title('Residual Plot')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Save the model for later use
import joblib

model_path = '/local/home/liubov_kurafeeva/CFDaPCR_intheloop/models/pcr_windspeed_model.pkl'
joblib.dump({
    'model': pcr_model,
    'simulation_lookup': simulation_lookup,
    'available_ws': available_ws,
    'sensor_in_coords': (sensor_in_x, sensor_in_y, sensor_in_z),
    'sequence_length': SEQUENCE_LENGTH,
    'train_metrics': {'rmse': train_rmse, 'r2': train_r2},
    'test_metrics': {'rmse': test_rmse, 'r2': test_r2}
}, model_path)

print(f"Model saved to: {model_path}")

In [ ]:
# Helper function to load model and make predictions
def load_and_predict(out_readings, model_path='/local/home/liubov_kurafeeva/CFDaPCR_intheloop/models/pcr_windspeed_model.pkl'):
    """
    Load the saved PCR model and predict windspeed at sensor_in.
    
    Parameters:
    -----------
    out_readings : array-like of shape (12,)
        12 consecutive windspeed readings from the 'out' sensor (in m/s)
    model_path : str
        Path to the saved model file
    
    Returns:
    --------
    dict
        Contains predicted U_Magnitude, matched simulation windspeed, and metadata
    """
    model_data = joblib.load(model_path)
    pcr = model_data['model']
    sim_lookup = model_data['simulation_lookup']
    avail_ws = model_data['available_ws']
    
    out_readings = np.array(out_readings).reshape(1, -1)
    prediction = pcr.predict(out_readings)[0]
    
    # Find which simulation was matched (based on 12th reading)
    ws_12th = out_readings[0, -1]
    matched_ws = min(avail_ws, key=lambda x: abs(x - ws_12th))
    
    return {
        'predicted_u_magnitude': prediction,
        'matched_simulation_ws': matched_ws,
        'sensor_in_coords': model_data['sensor_in_coords'],
        'sequence_length': model_data['sequence_length']
    }

# Test the helper function
test_result = load_and_predict(example_sequence)
print("Test of load_and_predict function:")
for key, value in test_result.items():
    print(f"  {key}: {value}")